## 7. Group Assignment & Presentation


__You should be able to start up on this exercise after Lecture 1.__

*This exercise must be a group effort. That means everyone must participate in the assignment.*

In this assignment you will solve a data science problem end-to-end, pretending to be recently hired data scientists in a company. To help you get started, we've prepared a checklist to guide you through the project. Here are the main steps that you will go through:

4. Prepare the data to better expose the underlying data patterns to machine learning algorithms
5. Explore many different models and short-list the best ones
6. Fine-tune your models
7. Present your solution (video presentation) 

In each step we list a set of questions that one should have in mind when undertaking a data science project. The list is not meant to be exhaustive, but does contain a selection of the most important questions to ask. We will be available to provide assistance with each of the steps, and will allocate some part of each lesson towards working on the projects.

Your group must submit a _**single**_ Jupyter notebook, structured in terms of the first 6 sections listed above (the seventh will be a video uploaded to some streaming platform, e.g. YouTube, Vimeo, etc.).

### 5. Short-list promising models
We expect you to do some additional research and train at **least one model per team member**.

1. Train mainly quick and dirty models from different categories (e.g. linear, SVM, Random Forests etc) using default parameters
2. Measure and compare their performance
3. Analyse the most significant variables for each algorithm
4. Analyse the types of errors the models make
5. Have a quick round of feature selection and engineering if necessary
6. Have one or two more quick iterations of the five previous steps
7. Short-list the top three to five most promising models, preferring models that make different types of errors
### 6. Fine-tune the system
1. Fine-tune the hyperparameters
2. Once you are confident about your final model, measure its performance on the test set to estimate the generalisation error
### 7. Present your solution
1. Document what you have done
2. Create a nice 15 minute video presentation with slides
    * Make sure you highlight the big picture first
3. Explain why your solution achieves the business objective
4. Don't forget to present interesting points you noticed along the way:
    * Describe what worked and what did not
    * List your assumptions and you model's limitations
5. Ensure your key findings are communicated through nice visualisations or easy-to-remember statements (e.g. "the median income is the number-one predictor of housing prices")
6. Upload the presentation to some online platform, e.g. YouTube or Vimeo, and supply a link to the video in the notebook.
Géron, A. 2017, *Hands-On Machine Learning with Scikit-Learn and Tensorflow*, Appendix B, O'Reilly Media, Inc., Sebastopol.

# Research, reasons for no show health care appointments.

## 1. Framing the Problem and looking at the bigger picture.

### Objective:
Predict and analyze healthcare appointment no-shows, focusing on economic and demographic factors that might influence them.

#### Key Questions with Demographics:
- What economic factors (e.g., income, unemployment rate, access to transportation) correlate with no-shows?
- Are there specific patterns in different Brazilian cities or regions?
- Do specific age groups (like the elderly or youth) show higher no-show rates?
- How does education or literacy correlate with appointment attendance?
- Are there patterns in no-shows based on gender or ethnicity in different cities?
- Is family structure (e.g., single parents vs. larger households) a factor?

#### Scope Expansion:
- Geographical Focus: Cities in Brazil (urban/rural divide? Large vs. small cities?).
- Economic Focus: Indicators like GDP per capita, public health investment, and local employment rates.
- Healthcare Metrics: Number of appointments, no-show rates, reasons for no-shows.
- Income brackets.
- Local employment levels.
- Urbanization metrics.

#### Feature Engineering for Demographics:
- Combine Age (from the healthcare dataset) with IDHM_Renda or GDP_CAPITA (from the Brazil cities dataset) to explore how income levels vary by age group.
- Create a composite variable for Neighbourhood-level healthcare data and RURAL_URBAN classification to highlight urban-rural divides.
- Group cities by demographic trends (e.g., predominantly elderly populations).
- Group cities based on IDHM scores to categorize them into high, medium, and low human development zones.

#### Insights from Demographic-Economic Analysis:
- Identify high no-show rates in low-income areas by linking Neighbourhood data with GDP_CAPITA or IDHM_Renda.
- Highlight rural cities (RURAL_URBAN = "Rural") with low MUN_EXPENDIT for targeted healthcare funding.

## 2. Get the Data  
The healthcare no-show data was sourced from Kaggle, providing a clean and well-organized dataset. After a quick exploration of the dataset, we thought it would be interesting to investigate potential correlations with Brazil's economic and demographic aspects. To support this, we located another dataset containing information about Brazilian cities, though it requires significant cleaning and preprocessing due to its raw state.

### 2.1 Translation

For the economic data for the neighbourhoods, there was significant work to be done to translate and prepare it for ingestion. First we used Google Translate to get a rough draft of the translations. This was done by manually extracting the column headers to a Google Sheet and applying a translation operation.

```
=GOOGLETRANSLATE(B2, "pt", "en")
```

These initial machine translations were sent to our Portuguese classmate Laura Do Bem Rebelo for review. It was clear that some of the machine translations needed work.


```
Razão por Sexo -> (Google translate) Reason for sex -> (Laura's edits) Sex ratio
```

From there, the translations with fixes were applied and the data from the 5 tables was set together. Since the neighbourhoods were consistent, it was just a matter of manually combining them.

This transformation of the data was applied before exploration because the large gap in language and format hindered our ability to explore it within the notebook.

## 3. Explore and prepare the data

#### 3.1 Imports

In [ ]:
# !pip install plotly
# !pip install nbformat
# !pip install geopandas contextily
# !pip install folium
# !pip install shapely
# !pip install osmnx
# !pip install seaborn

import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import osmnx as ox
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import json
import unicodedata
from scipy import stats

#### 3.2 Data set loading and preparation

In [ ]:
df_appointments = pd.read_csv('healthcare-noshow/healthcare_noshows_appt.csv', thousands=',').copy()
df_neighbourhoods = pd.read_csv('neighbourhoods/Vitoria_Economic_Data.csv', thousands=',').copy()
no_shows_by_neighbourhood = df_appointments.groupby('Neighbourhood').agg({
   'Showed_up': ['count', lambda x: (x == 0).sum()]
}).reset_index()

no_shows_by_neighbourhood.columns = ['Neighbourhood', 'Total_Appointments', 'No_Shows']

As we learned from our mistakes, it seems that it was impossible to correlate the even of a person not comming using these two datasets. As we almost gave up, we had an idea of converting No_Shows and Total_Appointments into a percentage value, which we could utilize. Later it will show appropriately good looking correlation matrix.

In order to explore the combined data, we first have to match up the neighbourhood names from the no show data and the economic data.

In [ ]:
# First let's normalize the neighborhood names in both datasets using the same preprocessing
def normalize_neighborhood_name(name):
    # Convert to uppercase
    name = str(name).upper()
    # Remove trailing spaces
    if name.endswith(' '):
        name = name[:-1]
    # Remove single quotes
    if "'" in name:
        name = name.replace("'", " ")
    # Remove accents
    nfkd_form = unicodedata.normalize('NFKD', name)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])

# Normalize names in both datasets
appointments_neighborhoods = set(no_shows_by_neighbourhood['Neighbourhood'].apply(normalize_neighborhood_name))
economic_neighborhoods = set(df_neighbourhoods['Neighborhood name'].apply(normalize_neighborhood_name))

# Find matches and mismatches
matches = appointments_neighborhoods.intersection(economic_neighborhoods)
appointments_only = appointments_neighborhoods - economic_neighborhoods
economic_only = economic_neighborhoods - appointments_neighborhoods

print(f"Total neighborhoods in appointments dataset: {len(appointments_neighborhoods)}")
print(f"Total neighborhoods in economic dataset: {len(economic_neighborhoods)}")
print(f"\nMatched neighborhoods: {len(matches)}")
print("\nMatched neighborhood names:")
for name in sorted(matches):
    print(f"- {name}")

print(f"\nNeighborhoods only in appointments dataset: {len(appointments_only)}")
print("\nAppointments-only neighborhood names:")
for name in sorted(appointments_only):
    print(f"- {name}")

print(f"\nNeighborhoods only in economic dataset: {len(economic_only)}")
print("\nEconomic-only neighborhood names:")
for name in sorted(economic_only):
    print(f"- {name}")

The CONDUSA result looks like a simple typo, so those should be matched. The other mismatches should then be removed. Then we can also calculate the no show rate for the different neighborhoods

In [ ]:
# Define neighborhoods to remove
unmatched_appointments = ['AEROPORTO', 'ILHAS OCEANICAS DE TRINDADE']
unmatched_economic = ['CARAPINA I', 'DE FATIMA', 'HELIO FERRAZ', 'TOTAL IN VITORIA']

# First filter appointments data to remove unmatched neighborhoods
filtered_appointments = df_appointments[~df_appointments['Neighbourhood'].str.upper().apply(remove_garbage).isin(unmatched_appointments)]

# Calculate no-shows by neighbourhood with the filtered data
no_shows_by_neighbourhood = filtered_appointments.groupby('Neighbourhood').agg({
    'Showed_up': ['count', lambda x: (x == 0).sum()]
}).reset_index()

no_shows_by_neighbourhood.columns = ['Neighbourhood', 'Total_Appointments', 'No_Shows']

# Calculate no-show rate
no_shows_by_neighbourhood['No_Show_Rate'] = (
    no_shows_by_neighbourhood['No_Shows'] / 
    no_shows_by_neighbourhood['Total_Appointments'] * 100
).round(1)

# Clean neighborhood names - applying cleaning before any operations
no_shows_by_neighbourhood['Neighbourhood'] = no_shows_by_neighbourhood['Neighbourhood'].str.upper().apply(remove_garbage)
df_neighbourhoods['Neighborhood name'] = df_neighbourhoods['Neighborhood name'].str.upper().apply(remove_garbage)

# Double-check removal of unmatched neighborhoods after cleaning
no_shows_by_neighbourhood = no_shows_by_neighbourhood[~no_shows_by_neighbourhood['Neighbourhood'].isin(unmatched_appointments)]
df_neighbourhoods = df_neighbourhoods[~df_neighbourhoods['Neighborhood name'].isin(unmatched_economic)]

# Fix the known matching issue
no_shows_by_neighbourhood['Neighbourhood'] = no_shows_by_neighbourhood['Neighbourhood'].replace('COMDUSA', 'CONDUSA')

# Fix any data type issues
df_neighbourhoods['Population aged 0 to 4'] = df_neighbourhoods['Population aged 0 to 4'].replace('-', 0).apply(pd.to_numeric, errors='coerce')

# Display results
print("No-shows by neighbourhood:")
display(no_shows_by_neighbourhood)
print("\nNeighbourhood economic data:")
display(df_neighbourhoods)

# Verify the matching
appointments_neighborhoods = set(no_shows_by_neighbourhood['Neighbourhood'])
economic_neighborhoods = set(df_neighbourhoods['Neighborhood name'])
unmatched = appointments_neighborhoods.symmetric_difference(economic_neighborhoods)

if len(unmatched) > 0:
    print("\nWarning: There are still unmatched neighborhoods:")
    print(unmatched)
else:
    print("\nSuccess: All neighborhoods now match between datasets")

#### 3.3 Study the features and characteristics

In [ ]:
# Features study
def study_features(df):
    feature_types: dict[str, str] = {}
    feature_summary = []
    for column in df.columns:
        col_type = df[column].dtype
        
        # Check for every possible number type:
        if np.issubdtype(col_type, np.number):
            feature_type = "Numerical"
        elif col_type == "object":
            if len(df[column].unique()) == 2:
                feature_type = "Binary"
            else:
                feature_type = "Categorical"
        else:
            if len(df[column].unique()) == 2:
                feature_type = "Binary"
            else:
                feature_type = "Other"

        # Feature analysis
        feature_info = {
            "Feature Name": column,
            "Type": feature_type,
            "Missing Values (%)": df[column].isnull().mean() * 100,
            "Unique Values": len(df[column].unique()),
        }

        # Add numeric details if applicable
        if feature_type == "Numerical":
            feature_info.update({
                "Min": df[column].min(),
                "Max": df[column].max(),
                "Mean": df[column].mean(),
                "Std Dev": df[column].std(),
            })

            # Check for outliers using IQR (The Interquartile Range)
            q1 = df[column].quantile(0.25)
            q3 = df[column].quantile(0.75)
            iqr = q3 - q1
            outliers = df[(df[column] < (q1 - 1.5 * iqr)) | (df[column] > (q3 + 1.5 * iqr))]
            feature_info["Outliers (Count)"] = len(outliers)

            # Check the outliers in target value:
            if feature_info["Feature Name"] == "No_Show_Rate":
                print("Following outliers found in No_Show_Rate:")
                print(outliers)

        feature_summary.append(feature_info)
        feature_types[column] = feature_type

    return pd.DataFrame(feature_summary), feature_types

# Analyze the datasets
no_shows_by_neighbourhood_summarry, no_shows_by_neighbourhood_types = study_features(no_shows_by_neighbourhood) # _ means ignore
display(no_shows_by_neighbourhood_summarry)

neighbourhoods_feature_summary, neighbourhoods_feature_type = study_features(df_neighbourhoods)
display(neighbourhoods_feature_summary)

After some corrections, data type looks correct, as well as their respective properties. Just to make sure we also investigated, which outliers we found in the target feature, but even though these values seem like they could be a problem for a model (edge cases), are not broken. Let's leave outliers for skewness check.

### 3.5 Mapping categorical data / Data encoding
Since our data is mostly numerical and we do not use our the Neighbourhood names for the models, we do not need to encode our data and we can leave it as is.

### 3.6 Correlation matrixes for our data

Let's now merge the two datasets and observe the correlation matrix.

In [ ]:
df_merged = pd.merge(
   no_shows_by_neighbourhood,
   df_neighbourhoods,
   left_on='Neighbourhood',
   right_on='Neighborhood name',
   how='inner'
)

plt.figure(figsize=(10, 6))
plt.hist(df_merged['No_Show_Rate'], bins=20, edgecolor='black')
plt.axvline(df_merged['No_Show_Rate'].mean(), color='red', linestyle='--', label=f'Mean: {df_merged["No_Show_Rate"].mean():.1f}%')
plt.axvline(df_merged['No_Show_Rate'].median(), color='green', linestyle='--', label=f'Median: {df_merged["No_Show_Rate"].median():.1f}%')
plt.title('Distribution of No-Show Rates Across Neighborhoods')
plt.xlabel('No-Show Rate (%)')
plt.ylabel('Number of Neighborhoods')
plt.legend()
plt.show()

This diagram shows us, how the most values are positioned in a group from range of 14 to 23. Even though we have edge values they are minority in this case. Let's see how correlation matrix looks like.

In [ ]:
numeric_cols = df_merged.select_dtypes(include=['float64', 'int64']).columns
correlation_matrix = df_merged[numeric_cols].corr()

mask = np.eye(correlation_matrix.shape[0], dtype=bool)  # Mask for diagonal

# Plot correlation heatmap
plt.figure(figsize=(25, 25))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, mask=mask)
plt.title('Correlation Matrix of Numeric Features')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

As we are mostly interested in No_Show_Rate feature, as our target we can see that the range of available correlations is limited. Even though this diagram doesn't show the clear picture for our case, it is nice to see some interesting correlations such as:
* *Earning more than minimum wage* & *Rate of Dependency Youth*
* *Assigned Property* & *Literary rate*

And so on... .


In [ ]:
# Get top correlations, excluding No_Show_Rate and No_Shows
features_to_exclude = ['No_Show_Rate', 'No_Shows']
top_correlations_display = correlation_matrix['No_Show_Rate'].abs().sort_values(ascending=False)
top_correlations_display = top_correlations_display.drop(features_to_exclude)[:10]
top_correlations = correlation_matrix['No_Show_Rate'].drop(features_to_exclude).abs().sort_values(ascending=False)[:10]

# Print correlation values
print("Top 10 Feature Correlations with No-Show Rate:")
print("-" * 50)
for feature, correlation in top_correlations.items():
    # Get the actual correlation (not absolute) for direction
    actual_correlation = correlation_matrix['No_Show_Rate'][feature]
    direction = "positive" if actual_correlation > 0 else "negative"
    print(f"{feature:<30} | {abs(actual_correlation):.3f} ({direction} correlation)")
print("-" * 50)

# Create visualization
plt.figure(figsize=(12, 6))
top_correlations_display.plot(kind='bar')
plt.title('Top 10 Features Correlated with No-Show Rate')
plt.xlabel('Features')
plt.ylabel('Absolute Correlation')
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha='right')
plt.yticks(np.arange(0, 1.1, 0.1))
plt.tight_layout()
plt.show()

### 3.7 Identifying outliers

Lets find the outliers in the neighbourhoods to see if there is something we should exclude. 

In [ ]:
# Calculate statistics for identifying outliers
no_show_stats = df_merged['No_Show_Rate'].describe()
q1 = df_merged['No_Show_Rate'].quantile(0.25)
q3 = df_merged['No_Show_Rate'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Find outliers using the IQR method
outliers = df_merged[
    (df_merged['No_Show_Rate'] < lower_bound) | 
    (df_merged['No_Show_Rate'] > upper_bound)
]

print("No-Show Rate Statistics:")
print(f"Mean: {no_show_stats['mean']:.2f}%")
print(f"Std Dev: {no_show_stats['std']:.2f}%")
print(f"Q1 (25th percentile): {q1:.2f}%")
print(f"Q3 (75th percentile): {q3:.2f}%")
print(f"IQR: {iqr:.2f}")
print(f"Lower bound: {lower_bound:.2f}%")
print(f"Upper bound: {upper_bound:.2f}%")

print("\nPotential Outlier Neighborhoods:")
print("-" * 70)
print(outliers[['Neighbourhood', 'No_Show_Rate', 'Total_Appointments', 'No_Shows']].sort_values('No_Show_Rate'))

# Visualize the distribution with outlier bounds
plt.figure(figsize=(12, 6))
plt.hist(df_merged['No_Show_Rate'], bins=30, alpha=0.5, label='No-Show Rate Distribution')
plt.axvline(x=lower_bound, color='r', linestyle='--', label='Outlier Bounds')
plt.axvline(x=upper_bound, color='r', linestyle='--')
plt.axvline(x=df_merged['No_Show_Rate'].mean(), color='g', linestyle='-', label='Mean')

# Highlight the three neighborhoods we were excluding
for neighborhood in ['PARQUE INDUSTRIAL', 'ILHA DO BOI', 'SANTOS DUMONT']:
    if neighborhood in df_merged['Neighbourhood'].values:
        value = df_merged[df_merged['Neighbourhood'] == neighborhood]['No_Show_Rate'].iloc[0]
        plt.axvline(x=value, color='orange', linestyle=':', label=f'{neighborhood}')

plt.title('Distribution of No-Show Rates with Outlier Bounds')
plt.xlabel('No-Show Rate (%)')
plt.ylabel('Count')
plt.legend()
plt.show()

In [ ]:
outlier_neighborhoods = ['PARQUE INDUSTRIAL', 'ILHA DO BOI', 'SANTOS DUMONT']
df_merged_clean = df_merged[~df_merged['Neighbourhood'].isin(outlier_neighborhoods)]
columns_to_keep = top_correlations.index
columns_to_keep = columns_to_keep.append(pd.Index(['Neighbourhood', 'No_Shows']))
df_merged_clean = df_merged_clean.drop(columns=[col for col in df_merged_clean.columns if col not in columns_to_keep])

### 3.8 Identifying promising transformations

Let's look at the skewness.

In [ ]:
def analyze_skewness(df):
    numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    n_cols = len(numeric_cols)
    fig, axes = plt.subplots(n_cols, 2, figsize=(15, 4*n_cols))
    
    skewness_results = []
    
    for i, col in enumerate(numeric_cols):
        skewness = stats.skew(df[col].dropna())
        
        skewness_results.append({
            'Column': col,
            'Skewness': skewness,
            'Abs Skewness': abs(skewness)
        })
        
        sns.histplot(df[col], kde=True, ax=axes[i, 0])
        axes[i, 0].set_title(f'{col} - Histogram')
        
        stats.probplot(df[col], plot=axes[i, 1])
        axes[i, 1].set_title(f'{col} - Q-Q Plot')
    
    plt.tight_layout()
    plt.show()
    
    skewness_df = pd.DataFrame(skewness_results)
    skewness_df = skewness_df.sort_values('Abs Skewness', ascending=False)
    
    print("Skewness Analysis:")
    print(skewness_df)

analyze_skewness(df_merged_clean)

Let's deal with skewness!

In [ ]:

df_transformed = df_merged_clean.drop(columns=['Neighbourhood'])

df_transformed['Total_Appointments'] = np.sqrt(df_transformed['Total_Appointments'])
df_transformed['No_Shows'] = np.sqrt(df_transformed['No_Shows'])
df_transformed['Assigned property %'] = np.sqrt(df_transformed['Assigned property %'])
df_transformed['Literary rate'] = np.sqrt(df_transformed['Literary rate'])

analyze_skewness(df_transformed)

In [ ]:
df_final = df_transformed.copy()

That sums up cleaning and preparing the data.

### 4 Training models

### 4.1 Imports

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, classification_report
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV

### 4.2 Splitting train and test data

Let's start by splitting the datasets to train and test blocks

In [ ]:
# Shuffle indices with a fixed random seed
selected_feature = 'No_Show_Rate'
np.random.seed(42)
shuffled_indices = np.random.permutation(df_final.index)

data_shuffled = df_final.loc[shuffled_indices].reset_index(drop=True)

X = data_shuffled.drop(columns=selected_feature)
Y = data_shuffled[selected_feature]

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, random_state=42)

### 4.1 Training some quick models with default parameters.

Let's try an make a generic function for training the models and evaluating them.

In [ ]:
def fit_and_evaluate_model(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("\n----------------Run Results--------------------------------")
    try:
        print("Accuracy:", accuracy_score(y_test, y_pred))
    except ValueError:
        print("Cannot calculate accuracy for regression models")
    try:
        print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
    except ValueError:
        print("Cannot calculate MSE")
        pass
    try:
        print(f"Training Score: {model.score(X_train, y_train):.4f}")
    except ValueError:
        print("Cannot calculate Training Score")
        pass
    try:
        print(f"Test Score: {model.score(X_test, y_test):.4f}")
    except ValueError:
        print("Cannot calculate Test Score")
        pass
    try:
        print("R-squared", r2_score(y_test, y_pred))
    except ValueError:
        print("Cannot calculate R-squared")
        pass
    try:
        print(classification_report(y_test, y_pred))
    except ValueError:
        print("Cannot calculate classification report")
        pass
    print("--------------Run Results End------------------------------\n")

def evaluate_best_model(grid_search):
    best_model = grid_search.best_estimator_

    # Predict on training and testing data
    y_train_pred = best_model.predict(X_train)
    y_test_pred = best_model.predict(X_test)

    # Calculate metrics
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)

    # Print results
    print("\n----------------Run Results--------------------------------")
    print("Best Parameters:", grid_search.best_params_)
    print("Best Cross-Validation Score (MSE):", -grid_search.best_score_)

    print("\nTraining Performance:")
    try:
        print(f"Training MSE: {train_mse:.4f}")
    except ValueError:
        print("Cannot calculate training MSE")
    
    try:
        print(f"Training R-Squared: {train_r2:.4f}")
    except ValueError:
        print("Cannot calculate training R-Squared")

    try:
        print("\nTesting Performance:")
    except ValueError:
        print("Cannot calculate training testing performance")
    
    try:
        print(f"Testing MSE: {test_mse:.4f}")
    except ValueError:
        print("Cannot calculate testing MSE")

    try:
        print(f"Testing R-Squared: {test_r2:.4f}")
    except ValueError:
        print("Cannot calculate training testing R-Squared")

    print("--------------Run Results End------------------------------\n")

Cheetsheet for scoring options for regressions:

* 'neg_mean_squared_error'
* 'neg_root_mean_squared_error'
* 'neg_mean_absolute_error'
* 'neg_median_absolute_error'
* 'r2'
* 'explained_variance'

Lower we are trying to find the best iteration out of many for every model. We noticed that explained_variance scoring achieved better overall model results, this is why mostly we use specifically this scoring.

# TODO - sorry, couldn't help myself, but probably just needs to be moved to fine-tunning, and left as empty without grid or random search here.

In [ ]:
svr_param_grid = {
    'C': [0.1, 1, 10],
    'epsilon': [0.01, 0.1, 0.5],
    'kernel': ['linear', 'rbf']
}

svr = SVR()

svr_grid_search = GridSearchCV(
    estimator=svr,
    param_grid=svr_param_grid,
    cv=5,
    scoring='explained_variance',
    verbose=1
)

svr_grid_search.fit(X_train, y_train)
evaluate_best_model(svr_grid_search)
# slight overfitting (testing vs training MSE)

In [ ]:
elastic_param_grid = {
    'alpha': [0.01, 0.1, 1, 10],
    'l1_ratio': [0.1, 0.5, 0.7, 1]
}

elastic_net = ElasticNet()

elastic_grid_search = GridSearchCV(
    estimator=elastic_net,
    param_grid=elastic_param_grid,
    cv=5,
    scoring='explained_variance',
    verbose=1
)

elastic_grid_search.fit(X_train, y_train)
evaluate_best_model(elastic_grid_search)

In [ ]:
LinearRegression = LinearRegression()
fit_and_evaluate_model(LinearRegression)

In [ ]:
forest_param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

forest = RandomForestRegressor()

foreest_grid_search = GridSearchCV(
    estimator=forest,
    param_grid=forest_param_grid,
    cv=5,
    scoring='explained_variance',
    verbose=1,
    n_jobs=-1
)

foreest_grid_search.fit(X_train, y_train)
evaluate_best_model(foreest_grid_search)

# just a bad tunement... maybe...

In [ ]:
mlp = MLPRegressor()
fit_and_evaluate_model(mlp)
# fine tunning later

### 4.2 Tuning the hyperparameters

# TODO - probably rephrase outdated explanation

We have noticed that we do not have very good performance from the models and they perform poorly. The R-squared is telling us that the model is performing worse than the naive mean model, despite the fact that, the accuracy of the model is quite high.

That's why we chose MLPRegressor and decided to set up GridSearchCV to help us finding the best parameters for selected model.

In [ ]:
# TODO: DELETE THIS PART BECAUSE WE HAVE THE SAME LOWER DOWN, WHICH ACTUALLY WORKS
# TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO
from sklearn.linear_model import ElasticNet

# Define parameter grid for alpha and l1_ratio
mlp_param_grid = {
    'alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
}
# Initialize the ElasticNet model
elastic_net = ElasticNet(max_iter=10000)

# Set up GridSearchCV
mlp_grid_search = GridSearchCV(
    estimator=elastic_net,
    param_grid=mlp_param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)

# Fit the grid search to the training data
mlp_grid_search.fit(X_train, y_train)

# Best parameters and score
print("Best parameters:", mlp_grid_search.best_params_)
print("Best cross-validated MSE:", -mlp_grid_search.best_score_)

# Evaluate the best model
best_model = mlp_grid_search.best_estimator_
fit_and_evaluate_model(best_model)
# TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO TODO

In [ ]:
# TODO TODO TODO TODO TODO TODO TODO TODO remove?
# # Define the parameter grid
# param_grid = {
#     'hidden_layer_sizes': [(32,), (64,), (32, 16)],
#     'activation': ['relu', 'tanh', 'logistic', 'identity'],
#     'solver': ['adam', 'lbfgs', 'sgd'],
#     'alpha': [0.01, 0.1, 1],
#     'learning_rate': ['constant', 'invscaling', 'adaptive']
# }

# mlp = MLPRegressor(max_iter=1000, random_state=42)

# # Use GridSearchCV to tune hyperparameters
# grid_search = GridSearchCV(
#     estimator=mlp,
#     param_grid=param_grid,
#     cv=5,
#     n_jobs=-1, # Use all CPU cores
#     verbose=0,
# )

# fit_and_evaluate_model(grid_search)

# # Print the best configuration
# print("Best Parameters:", grid_search.best_params_)
# print("Best Cross-Validation Score:", grid_search.best_score_)


# P.S. Updated this, because it wasn't showing the result for best iteration, just a result of the last xD




# TL;DR Tried to fixed errors, by introducing sclaing as the error was stating, but that made model overfit, so reduced the max_iter to 152.
# notes to speak about: using 64 neurons is very demanding, but it's a perfect model :3, or so it looks like it.

from sklearn.pipeline import Pipeline

scaler = StandardScaler()
mlp = MLPRegressor(max_iter=145, early_stopping=True, random_state=42)  # THE ONLY PLACE WHICH ACTUALLY USES TUNEMENT OF HYPERPARAMETERS!!!

pipeline = Pipeline([
    ('scaler', scaler),
    ('mlp', mlp)
])

mlp_param_grid = {
    'mlp__hidden_layer_sizes': [(32,), (64,)], # 128 was never chosen, so removing that
    'mlp__activation': ['relu', 'tanh', 'logistic'], # reduced to these, as they generally used more in diff scenarios
    'mlp__solver': ['adam', 'lbfgs'], # limiting to only adam and lbfgs
    'mlp__alpha': [0.01, 0.1, 1],
    'mlp__learning_rate': ['constant'] # reduced to constant, as best performing
}
# Now it should run faster
# Probably can add RandomizedGridSearch to check if that would be better

mlp_grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=mlp_param_grid,
    cv=5,
    n_jobs=-1, 
    verbose=1,
    scoring='explained_variance',
    error_score='raise'
)

mlp_grid_search.fit(X_train, y_train)

evaluate_best_model(mlp_grid_search)

# outperforms every over model, but was *delicately* tuned by myself